# Tahoe Culvert Layer + Condition Table

**PROTECT Task 3.3 - Assets (culverts)**

Engineers a basin-wide **Culvert** point feature class and a related **1:many condition/inspection table** from jurisdiction-supplied source data in `C:\GIS\Culvert`, and writes them to a File GDB (plus a GeoPackage fallback) with a relationship class.

## Design decisions (confirmed)
- **Output:** File GDB - `Culverts` point feature class + standalone `CulvertCondition` table, related by `culvert_id` (`ONE_TO_MANY`).
- **Condition model:** 1:many. A culvert can carry many dated inspection records. Each record keeps the jurisdiction's **raw rating** plus a `condition_scheme` describing that jurisdiction's scale - no cross-walking of ratings yet.
- **ID strategy:** keep each jurisdiction's original ID (`source_id`) and add a `jurisdiction` field. Stable key: `culvert_id = "<jurisdiction>|<source_id>"`.
- **Lines to points:** pipe centerlines (El Dorado, Washoe) are collapsed to a single representative point per `cfg["run"]["line_to_point"]` (midpoint / start / end). `geom_source` records how each point was derived.

## Source inventory (as delivered, July 2026)

| Jurisdiction | Delivered | Geometry | Condition data |
|---|---|---|---|
| Placer County | `Tahoe Culverts.gdb` / `Tahoe_Culverts_7_17_2026` | 660 points | 1-5 rating + inspect date, same row |
| Douglas County | `Stormwater.gdb` / `Stormwater_Inspections_Rev_3` | 2,336 inspection points (1,858 assets) | Maintenance score 1-5, repeat visits |
| El Dorado County | `SW_PIPE.gdb` / `SW_PIPE` | 2,707 pipe lines | none delivered |
| Washoe County | `ConveyancePipe.shp` + `Condition.dbf` | 2,413 pipe lines | separate table, `asset_guid` -> `GlobalID` |
| Caltrans D3 | 5 xlsx drainage inventories (ED-50, ED-89, PLA-28, PLA-89, PLA-267) | inlet/outlet lat-lon per segment | last-inspection date only |
| CSLT | AGOL Stormwater FeatureServer, layer 415006 `Culvert` | 40 culvert lines | none (no culvert inspection table on the service) |
| NDOT | none - data sharing agreement pending | - | - |
| TRPA Legacy | `PROTECT_analysis/Assets.gdb/Tahoe_Culvert` (read-only, prior compilation incl. USFS) | 2,287 points; unmatched ones appended as provisional | Good/Fair/Poor text on ~350 records |

In [1]:
import sys
print("Python:", sys.executable)

import logging
import warnings
from pathlib import Path

import yaml
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely import wkb
from shapely.geometry import Point, MultiLineString
from shapely.ops import linemerge

warnings.filterwarnings("ignore")

# arcpy is only needed for the GDB write. Import lazily so everything else
# still runs when no ArcGIS license is available (e.g. Pro holds it).
try:
    import arcpy
    HAS_ARCPY = True
except Exception as e:
    HAS_ARCPY = False
    print("arcpy unavailable - GDB step will be skipped:", e)

# Repo root (works whether the kernel starts in the repo root or notebooks/)
REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent

with open(REPO / "config.yaml") as f:
    cfg = yaml.safe_load(f)


def get_logger(name):
    logs = REPO / "logs"
    logs.mkdir(exist_ok=True)
    # pd.Timestamp, not datetime: a successful arcpy import shadows the datetime name
    ts = pd.Timestamp.now().strftime("%Y-%m-%d_%H%M%S")
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    logger.handlers.clear()  # avoid duplicate handlers on notebook re-run
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s", "%Y-%m-%d %H:%M:%S")
    for h in (logging.StreamHandler(sys.stdout), logging.FileHandler(logs / f"{name}_{ts}.log")):
        h.setFormatter(fmt)
        logger.addHandler(h)
    return logger


log = get_logger("culvert_layer")

RAW = Path(cfg["paths"]["raw_root"])          # C:\GIS\Culvert
PROCESSED = REPO / cfg["paths"]["processed"]
OUTPUTS = REPO / cfg["paths"]["outputs"]
PROCESSED.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)

TARGET_CRS = f"EPSG:{cfg['output']['target_epsg']}"
LINE_TO_POINT = cfg["run"]["line_to_point"]
LOAD_DATE = pd.Timestamp.today().normalize()

log.info(f"Raw root: {RAW}")
log.info(f"Target CRS: {TARGET_CRS} | line_to_point: {LINE_TO_POINT}")

Python: C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\python.exe


2026-07-29 16:10:52 | INFO | Raw root: C:\GIS\Culvert


2026-07-29 16:10:52 | INFO | Target CRS: EPSG:26910 | line_to_point: midpoint


## Target schema
One row per culvert in `Culverts`; one row per inspection in `CulvertCondition`. Ratings stay raw per jurisdiction - `condition_scheme` documents each scale so they can be normalized later, deliberately.

In [2]:
# (field_name, arcpy_type, length_or_None). Order = column order in the GDB.
CULVERT_FIELDS = [
    ("jurisdiction",   "TEXT",   60),
    ("source_id",      "TEXT",   100),
    ("culvert_id",     "TEXT",   160),   # "<jurisdiction>|<source_id>" - stable key
    ("feature_type",   "TEXT",   40),    # culvert / stormwater pipe
    ("road_name",      "TEXT",   254),
    ("material",       "TEXT",   60),
    ("xsection_shape", "TEXT",   80),   # legacy source has verbose shape descriptions
    ("span_in",        "DOUBLE", None),  # diameter (round) or span, inches
    ("rise_in",        "DOUBLE", None),
    ("length_ft",      "DOUBLE", None),
    ("inlet_type",     "TEXT",   60),
    ("outlet_type",    "TEXT",   60),
    ("install_year",   "SHORT",  None),
    ("comments",       "TEXT",   500),
    ("geom_source",    "TEXT",   40),    # point / line-midpoint / inlet-latlon ...
    ("data_source",    "TEXT",   160),
    ("load_date",      "DATE",   None),
]

CONDITION_FIELDS = [
    ("culvert_id",       "TEXT",   160),  # FK -> Culverts.culvert_id
    ("jurisdiction",     "TEXT",   60),
    ("inspection_date",  "DATE",   None),
    ("condition_rating", "TEXT",   30),   # RAW value from the jurisdiction
    ("condition_scheme", "TEXT",   120),  # what the raw value means
    ("structural_cond",  "TEXT",   60),
    ("blockage_pct",     "DOUBLE", None),
    ("maintenance_need", "TEXT",   60),
    ("inspector",        "TEXT",   120),
    ("notes",            "TEXT",   500),
    ("data_source",      "TEXT",   160),
    ("load_date",        "DATE",   None),
]

CULVERT_COLS = [f[0] for f in CULVERT_FIELDS]
CONDITION_COLS = [f[0] for f in CONDITION_FIELDS]


def std_culverts(df: pd.DataFrame, geometry, src_crs=None) -> gpd.GeoDataFrame:
    """Coerce a partial frame to the full culvert schema + geometry in TARGET_CRS."""
    out = df.copy()
    for c in CULVERT_COLS:
        if c not in out.columns:
            out[c] = None
    out = out[CULVERT_COLS]
    g = gpd.GeoDataFrame(out, geometry=geometry, crs=src_crs or TARGET_CRS)
    if src_crs and str(src_crs) != TARGET_CRS:
        g = g.to_crs(TARGET_CRS)
    return g


def std_condition(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in CONDITION_COLS:
        if c not in out.columns:
            out[c] = None
    return out[CONDITION_COLS]


def dedupe_ids(s: pd.Series) -> pd.Series:
    """Make IDs unique by suffixing -2, -3... on repeats."""
    s = s.astype(str)
    counts = s.groupby(s).cumcount()
    return s.where(counts == 0, s + "-" + (counts + 1).astype(str))


def line_rep_point(geom, how=None):
    """Collapse a (Multi)LineString to a representative Point."""
    how = how or LINE_TO_POINT
    if geom is None or geom.is_empty:
        return None
    if isinstance(geom, MultiLineString):
        merged = linemerge(geom)
        if isinstance(merged, MultiLineString):  # disjoint parts: take longest
            merged = max(merged.geoms, key=lambda g: g.length)
        geom = merged
    if how == "start":
        return Point(geom.coords[0])
    if how == "end":
        return Point(geom.coords[-1])
    return geom.interpolate(0.5, normalized=True)  # midpoint


def force_2d(geom):
    if geom is None:
        return None
    return wkb.loads(wkb.dumps(geom, output_dimension=2))


def num(s):
    return pd.to_numeric(s, errors="coerce")

## Placer County
`Tahoe_Culverts_7_17_2026` - already culvert points (EPSG:2226). Condition (1-5) and inspection date ride on the same row, so each feature yields one culvert and one condition record. 9 `STID_PM` values repeat (multi-culvert sites) - suffixed to stay unique.

In [3]:
src = RAW / "PLACER" / "Tahoe Culverts.gdb"
p = gpd.read_file(src, layer="Tahoe_Culverts_7_17_2026")
log.info(f"[Placer] read {len(p)} rows")

p["source_id"] = dedupe_ids(p["STID_PM"])
placer_culverts = std_culverts(
    pd.DataFrame({
        "jurisdiction": "Placer County",
        "source_id": p["source_id"],
        "culvert_id": "Placer County|" + p["source_id"],
        "feature_type": "culvert",
        "road_name": p["STREET_NAME"],
        "material": p["Pipe_Material"],
        "span_in": num(p["Diameter"]).fillna(num(p["Size__in1"])),
        "length_ft": num(p["LENGTH__FT_"]),
        "install_year": pd.to_datetime(p["INSTALL_DATE"], errors="coerce").dt.year,
        "comments": p["Comments_1"],
        "geom_source": "point",
        "data_source": "Tahoe Culverts.gdb/Tahoe_Culverts_7_17_2026",
        "load_date": LOAD_DATE,
    }),
    geometry=p.geometry.apply(force_2d),
    src_crs=p.crs,
)

placer_condition = std_condition(pd.DataFrame({
    "culvert_id": "Placer County|" + p["source_id"],
    "jurisdiction": "Placer County",
    "inspection_date": pd.to_datetime(p["INSPECT_DATE"], errors="coerce"),
    "condition_rating": p["Condition"],
    "condition_scheme": "Placer 1-5 rating (semantics unconfirmed)",
    "blockage_pct": num(p["Percent_Obstructed_12"]),
    "maintenance_need": p["Maintenance_Required"],
    "inspector": p["CheckedBy"],
    "notes": p["Comments_1"],
    "data_source": "Tahoe Culverts.gdb/Tahoe_Culverts_7_17_2026",
    "load_date": LOAD_DATE,
}))
log.info(f"[Placer] {len(placer_culverts)} culverts, {len(placer_condition)} condition records")

2026-07-29 16:10:53 | INFO | [Placer] read 660 rows


2026-07-29 16:10:53 | INFO | [Placer] 660 culverts, 660 condition records


## Douglas County
`Stormwater_Inspections_Rev_3` - inspection-centric points (EPSG:4326) covering all drain features. Filtered to `DrainFeature == '1 - Culvert'`. `GID_Join` is the asset key; repeat visits make this genuinely 1:many. The asset row is the most recent inspection (preferring `CurrentInspection == 'Yes'`); every visit becomes a condition record. Condition = maintenance score (1 = needs replacement ... 5 = none needed).

In [4]:
src = RAW / "DOUGLAS" / "Stormwater" / "Stormwater.gdb"
d = gpd.read_file(src, layer="Stormwater_Inspections_Rev_3")
d = d[d["DrainFeature"] == "1 - Culvert"].copy()
# 30 rows lack GID_Join; AreaID (near-unique per visit) stands in as the asset key
n_nokey = int(d["GID_Join"].isna().sum())
if n_nokey:
    log.warning(f"[Douglas] {n_nokey} rows have no GID_Join - keyed by AreaID instead")
d["GID_Join"] = d["GID_Join"].fillna("AREA-" + d["AreaID"].astype(str))
log.info(f"[Douglas] {len(d)} culvert inspection rows, {d['GID_Join'].nunique()} unique assets")

d["CreationDate"] = pd.to_datetime(d["CreationDate"], errors="coerce", utc=True).dt.tz_localize(None)
d["_cur"] = (d["CurrentInspection"] == "Yes").astype(int)
d = d.sort_values(["GID_Join", "_cur", "CreationDate"])
assets = d.groupby("GID_Join", as_index=False).last()  # latest / current visit per asset

douglas_culverts = std_culverts(
    pd.DataFrame({
        "jurisdiction": "Douglas County",
        "source_id": assets["GID_Join"],
        "culvert_id": "Douglas County|" + assets["GID_Join"],
        "feature_type": "culvert",
        "road_name": assets["LandmarkAddress"],
        "material": assets["CulvertMaterial"].replace("N/A", None),
        "span_in": num(assets["CulvertSizeText"]),
        "comments": ("Area " + assets["Area"].astype(str) + " / " + assets["AreaID"].astype(str)
                     + " | County maintained: " + assets["CountyMaintained"].astype(str)),
        "geom_source": "point",
        "data_source": "Stormwater.gdb/Stormwater_Inspections_Rev_3",
        "load_date": LOAD_DATE,
    }),
    geometry=gpd.GeoSeries(assets["geometry"]).apply(force_2d),
    src_crs=d.crs,
)

douglas_condition = std_condition(pd.DataFrame({
    "culvert_id": "Douglas County|" + d["GID_Join"],
    "jurisdiction": "Douglas County",
    "inspection_date": d["CreationDate"],
    "condition_rating": num(d["Maintenance_Score_Code"]).astype("Int64").astype(str).replace("<NA>", None),
    "condition_scheme": "Douglas maintenance score 1-5 (1=replacement, 5=none needed)",
    "maintenance_need": d["MaintenanceScoreText"],
    "inspector": d["Creator"],
    "notes": d["Notes"],
    "data_source": "Stormwater.gdb/Stormwater_Inspections_Rev_3",
    "load_date": LOAD_DATE,
}))
log.info(f"[Douglas] {len(douglas_culverts)} culverts, {len(douglas_condition)} condition records")

2026-07-29 16:10:53 | WARNING | [Douglas] 30 rows have no GID_Join - keyed by AreaID instead


2026-07-29 16:10:53 | INFO | [Douglas] 858 culvert inspection rows, 858 unique assets


2026-07-29 16:10:53 | INFO | [Douglas] 858 culverts, 858 condition records


## El Dorado County
`SW_PIPE` - stormwater pipe centerlines (EPSG:2226), collapsed to representative points. **No condition data delivered.** Attribute domains (`MATL_CODE`, `TYPE_SHAPE_CODE`, inlet/outlet treatment codes) are undocumented - the README in the delivery is empty - so raw codes are carried as `code N` until the county supplies the lookup. Pipes retired via `STOP_DATE_CODE == 1` are dropped. Tagged `feature_type = 'stormwater pipe'` because culverts are not distinguished from conveyance pipe in this inventory.

In [5]:
src = RAW / "ELDORADO" / "SW_PIPE.gdb"
e = gpd.read_file(src, layer="SW_PIPE")
n0 = len(e)
e = e[e["STOP_DATE_CODE"] != 1].copy()   # drop retired pipes
log.info(f"[El Dorado] read {n0} rows, {len(e)} active after STOP_DATE_CODE filter")

e["source_id"] = dedupe_ids(e["PIPE_FEAT_ID"].astype("Int64").astype(str))


def code_txt(s):
    v = num(s)
    return v.map(lambda x: None if pd.isna(x) else f"code {int(x)}")


eldorado_culverts = std_culverts(
    pd.DataFrame({
        "jurisdiction": "El Dorado County",
        "source_id": e["source_id"],
        "culvert_id": "El Dorado County|" + e["source_id"],
        "feature_type": "stormwater pipe",
        "road_name": e["LOC_DESCR"],
        "material": code_txt(e["MATL_CODE"]),
        "xsection_shape": code_txt(e["TYPE_SHAPE_CODE"]),
        "span_in": num(e["MAJOR_DIM"]),
        "length_ft": num(e["Shape_Length"]),          # EPSG:2226 is US-feet
        "inlet_type": code_txt(e["INLET_TREATMENT_CODE"]),
        "outlet_type": code_txt(e["OUTLET_TREATMENT_CODE"]),
        "install_year": num(e["YR_BUILT"]).astype("Int64"),
        "geom_source": f"line-{LINE_TO_POINT}",
        "data_source": "SW_PIPE.gdb/SW_PIPE",
        "load_date": LOAD_DATE,
    }),
    geometry=e.geometry.apply(line_rep_point),
    src_crs=e.crs,
)
eldorado_condition = std_condition(pd.DataFrame(columns=CONDITION_COLS))
log.info(f"[El Dorado] {len(eldorado_culverts)} pipe points, 0 condition records (none delivered)")

2026-07-29 16:10:54 | INFO | [El Dorado] read 2707 rows, 2658 active after STOP_DATE_CODE filter


2026-07-29 16:10:54 | INFO | [El Dorado] 2658 pipe points, 0 condition records (none delivered)


## Washoe County
`ConveyancePipe.shp` (lines, EPSG:3423 ft-US) + separate `Condition.dbf`. Condition joins `asset_guid` -> pipe `GlobalID`; only `asset_Type == 'ConveyancePipe'` rows apply (806 of 1,716 - the rest describe ditches, basins, traps not in this layer). The delivered `condition` field is uniformly 0, so the meaningful signals are the structural / blockage subscores and `perc_full`. Length comes from the projected geometry because the GNSS length fields are zeros.

In [6]:
wdir = RAW / "WASHOE" / "TRPADataRequest20260714" / "TRPADataRequest20260714"
w = gpd.read_file(wdir / "ConveyancePipe.shp")
log.info(f"[Washoe] read {len(w)} conveyance pipes")

w["source_id"] = dedupe_ids(w["NTCD_Asset"])
w["_len_ft"] = w.geometry.length          # EPSG:3423 is US-feet
w["_align"] = w["Pipe_Align"].fillna("Unknown")

washoe_culverts = std_culverts(
    pd.DataFrame({
        "jurisdiction": "Washoe County",
        "source_id": w["source_id"],
        "culvert_id": "Washoe County|" + w["source_id"],
        # Cross/Driveway alignments function as culverts; rest is conveyance pipe
        "feature_type": np.where(w["_align"].isin(["Cross", "Driveway"]),
                                 "culvert", "stormwater pipe"),
        "road_name": w["Road_Name"],
        "material": w["Asset_Subt"],
        "span_in": num(w["Pipe_Dia_i"]),
        "length_ft": w["_len_ft"].round(1),
        "comments": ("Alignment: " + w["_align"] + " | WCasset: " + w["WCasset_Nm"].astype(str)),
        "geom_source": f"line-{LINE_TO_POINT}",
        "data_source": "ConveyancePipe.shp",
        "load_date": LOAD_DATE,
    }),
    geometry=w.geometry.apply(line_rep_point),
    src_crs=w.crs,
)

wc = gpd.read_file(wdir / "Condition.dbf")
wc = wc[wc["asset_Type"] == "ConveyancePipe"].copy()
guid_to_id = dict(zip(w["GlobalID"], w["source_id"]))
wc["source_id"] = wc["asset_guid"].map(guid_to_id)
matched = wc["source_id"].notna()
if (~matched).sum():
    log.warning(f"[Washoe] {(~matched).sum()} condition rows have no matching pipe GlobalID - dropped")
wc = wc[matched]

washoe_condition = std_condition(pd.DataFrame({
    "culvert_id": "Washoe County|" + wc["source_id"],
    "jurisdiction": "Washoe County",
    "inspection_date": pd.to_datetime(wc["insp_date"], errors="coerce"),
    "condition_rating": wc["Structural"].astype(str),
    "condition_scheme": "Washoe structural subscore (coded; semantics unconfirmed)",
    "structural_cond": "Structural " + wc["Structural"].astype(str) + ", Blockage " + wc["Blockage"].astype(str),
    "blockage_pct": num(wc["perc_full"]),
    "maintenance_need": wc["Update_Typ"],
    "notes": wc["note"],
    "data_source": "Condition.dbf",
    "load_date": LOAD_DATE,
}))
log.info(f"[Washoe] {len(washoe_culverts)} pipes, {len(washoe_condition)} condition records")

2026-07-29 16:10:55 | INFO | [Washoe] read 2413 conveyance pipes


2026-07-29 16:10:55 | INFO | [Washoe] 2413 pipes, 806 condition records


## Caltrans District 3
Five xlsx drainage inventories (one per route segment). Each row is a culvert *segment* between end treatments; segments group into a physical crossing by `SYSNO`. One culvert per `SYSNO`: geometry at the first segment's inlet lat-lon, lengths summed, one condition record per distinct inspection date (date only - Caltrans supplied no rating).

In [7]:
cdir = RAW / "CALTRANS"
frames = []
for xlsx in sorted(cdir.glob("*.xlsx")):
    df = pd.read_excel(xlsx, sheet_name=0)
    df["__file"] = xlsx.name
    frames.append(df)
    log.info(f"[Caltrans] {xlsx.name}: {len(df)} segment rows")
ct = pd.concat(frames, ignore_index=True)
n_blank = int(ct["SYSNO"].isna().sum())
if n_blank:
    log.warning(f"[Caltrans] dropped {n_blank} blank row(s) with no SYSNO")
    ct = ct[ct["SYSNO"].notna()].copy()

C = {  # header shorthand (headers contain line breaks)
    "insp": "Culvert\nLast Inspection", "pm": "Culvert\nPostmile",
    "shape": "Culvert\nShape", "matl": "Culvert\nMaterial",
    "dia": "Culvert\nDiameter (FT)", "wid": "Culvert\nWidth (FT)",
    "hgt": "Culvert\nHeight (FT)", "length": "Culvert\nMeasured Length (ft)",
    "lat": "Inlet\nLatitude", "lon": "Inlet\nLongitude",
}
ct["SYSNO"] = ct["SYSNO"].astype(str)

rows, cond_rows = [], []
for sysno, g in ct.groupby("SYSNO", sort=False):
    first = g.iloc[0]
    span = num(pd.Series([first[C["dia"]]])).iloc[0]
    width = num(pd.Series([first[C["wid"]]])).iloc[0]
    height = num(pd.Series([first[C["hgt"]]])).iloc[0]
    rows.append({
        "jurisdiction": "Caltrans",
        "source_id": sysno,
        "culvert_id": f"Caltrans|{sysno}",
        "feature_type": "culvert",
        "road_name": f"SR-{first['Route']} PM {first[C['pm']]} ({first['County']})",
        "material": first[C["matl"]],
        "xsection_shape": first[C["shape"]],
        "span_in": (span if span else width or None) and float((span if span else width)) * 12.0,
        "rise_in": float(height) * 12.0 if height else None,
        "length_ft": num(g[C["length"]]).sum(),
        "comments": f"{len(g)} segment(s); file {first['__file']}",
        "geom_source": "inlet-latlon",
        "data_source": first["__file"],
        "load_date": LOAD_DATE,
        "_lat": num(g[C["lat"]]).dropna().iloc[0] if num(g[C["lat"]]).notna().any() else None,
        "_lon": num(g[C["lon"]]).dropna().iloc[0] if num(g[C["lon"]]).notna().any() else None,
    })
    for insp in pd.to_datetime(g[C["insp"]], errors="coerce").dropna().dt.normalize().unique():
        cond_rows.append({
            "culvert_id": f"Caltrans|{sysno}",
            "jurisdiction": "Caltrans",
            "inspection_date": pd.Timestamp(insp),
            "condition_scheme": "Caltrans - inspection date only, no rating delivered",
            "data_source": first["__file"],
            "load_date": LOAD_DATE,
        })

ctdf = pd.DataFrame(rows)
geom = [Point(lon, lat) if pd.notna(lat) and pd.notna(lon) else None
        for lat, lon in zip(ctdf.pop("_lat"), ctdf.pop("_lon"))]
caltrans_culverts = std_culverts(ctdf, geometry=geom, src_crs="EPSG:4326")
caltrans_condition = std_condition(pd.DataFrame(cond_rows))
log.info(f"[Caltrans] {len(caltrans_culverts)} culverts (from {len(ct)} segments), "
         f"{len(caltrans_condition)} condition records")

2026-07-29 16:10:58 | INFO | [Caltrans] ed-050 pm 66.5-80.4 ct drainage.xlsx: 650 segment rows


2026-07-29 16:10:59 | INFO | [Caltrans] ed-089 pm 0.0-27.4 ct drainage.xlsx: 514 segment rows


2026-07-29 16:10:59 | INFO | [Caltrans] pla-028 pm 0.0-11.03 ct drainage.xlsx: 692 segment rows


2026-07-29 16:10:59 | INFO | [Caltrans] pla-089 pm 0.0-12.2 ct drainage.xlsx: 694 segment rows


2026-07-29 16:10:59 | INFO | [Caltrans] pla-267 pm 6.6-9.9 ct drainage.xlsx: 82 segment rows


2026-07-29 16:10:59 | WARNING | [Caltrans] dropped 12 blank row(s) with no SYSNO


2026-07-29 16:11:00 | INFO | [Caltrans] 698 culverts (from 2620 segments), 811 condition records


## City of South Lake Tahoe
Live AGOL FeatureServer; layer `415006 Culvert` (40 culvert lines). Coded domains are decoded from the service definition. Lines collapse to representative points like the other line sources. The service has inspection tables for outfalls/catch basins/BMPs/manholes but **none for culverts**, so no condition records. Abandoned/retired/removed culverts are dropped by `lifecyclestatus`.

In [8]:
import requests

CSLT_URL = cfg["sources"]["cslt_culvert_layer"]

layer_def = requests.get(CSLT_URL, params={"f": "pjson"}, timeout=60).json()
domains = {f["name"]: {cv["code"]: cv["name"] for cv in f["domain"]["codedValues"]}
           for f in layer_def["fields"]
           if f.get("domain") and f["domain"].get("codedValues")}

gj = requests.get(f"{CSLT_URL}/query",
                  params={"where": "1=1", "outFields": "*", "f": "geojson"},
                  timeout=120).json()
s = gpd.GeoDataFrame.from_features(gj["features"], crs="EPSG:4326")
log.info(f"[CSLT] fetched {len(s)} culverts from FeatureServer")

# drop abandoned/retired/removed
RETIRED = {32, 64, 128}
s = s[~s["lifecyclestatus"].isin(RETIRED)].copy()

s["source_id"] = dedupe_ids(s["assetid"].fillna(s["objectid"].astype(str)))
s["_len_ft"] = s.to_crs(TARGET_CRS).geometry.length * 3.28084


def dom(col):
    return s[col].map(domains.get(col, {}))


cslt_culverts = std_culverts(
    pd.DataFrame({
        "jurisdiction": "City of South Lake Tahoe",
        "source_id": s["source_id"],
        "culvert_id": "City of South Lake Tahoe|" + s["source_id"],
        "feature_type": "culvert",
        "material": dom("material"),
        "xsection_shape": dom("pipeshape"),
        "span_in": num(s["diameter"]).replace(0, np.nan).fillna(num(s["width"]) * 12),
        "rise_in": num(s["height"]).replace(0, np.nan) * 12,
        "length_ft": num(s["linear_feet"]).replace(0, np.nan).fillna(s["_len_ft"].round(1)),
        # bound-check epoch-ms before converting: the live service has at least one
        # absurd installdate that overflows pandas' unit="ms" conversion
        "install_year": pd.to_datetime(
            num(s["installdate"]).where(lambda v: v.abs() < 5e12),
            unit="ms", errors="coerce").dt.year,
        "comments": ("Type: " + dom("assettype").astype(str) + " | Owned by: "
                     + dom("ownedby").astype(str) + " | Status: " + dom("lifecyclestatus").astype(str)),
        "geom_source": f"line-{LINE_TO_POINT}",
        "data_source": "CSLT Stormwater FeatureServer/415006",
        "load_date": LOAD_DATE,
    }),
    geometry=s.geometry.apply(line_rep_point),
    src_crs="EPSG:4326",
)
cslt_condition = std_condition(pd.DataFrame(columns=CONDITION_COLS))
log.info(f"[CSLT] {len(cslt_culverts)} culverts, 0 condition records (no culvert inspection table)")

2026-07-29 16:11:01 | INFO | [CSLT] fetched 40 culverts from FeatureServer


2026-07-29 16:11:01 | INFO | [CSLT] 32 culverts, 0 condition records (no culvert inspection table)


## NDOT - pending
No data yet; the sensitive/restricted data sharing agreement is still in signature. Slot a reader in when the delivery arrives (state highway culverts on US-50, SR-28, SR-207, SR-431).

## Combine and clip to the TRPA boundary
Stack all jurisdictions, enforce key uniqueness, clip to the TRPA boundary (layer 4 of the TRPA Boundaries MapServer), and persist intermediates (GeoPackage + parquet/CSV) regardless of whether the GDB step can run. Features outside the boundary (Caltrans postmile ranges, Washoe T-areas, El Dorado pipes beyond the basin) are dropped and listed in `outputs/clipped_out_of_basin.csv`; their condition records go with them.

In [9]:
culverts = pd.concat(
    [placer_culverts, douglas_culverts, eldorado_culverts, washoe_culverts,
     caltrans_culverts, cslt_culverts],
    ignore_index=True)
culverts = gpd.GeoDataFrame(culverts, geometry="geometry", crs=TARGET_CRS)

condition = pd.concat(
    [placer_condition, douglas_condition, eldorado_condition, washoe_condition,
     caltrans_condition, cslt_condition],
    ignore_index=True)

dups = culverts["culvert_id"].duplicated().sum()
if dups:
    log.warning(f"{dups} duplicate culvert_id after combine - keeping first")
    culverts = culverts.drop_duplicates("culvert_id", keep="first").reset_index(drop=True)

# --- Clip to the TRPA boundary ---
bj = requests.get(f"{cfg['sources']['trpa_boundary_layer']}/query",
                  params={"where": "1=1", "outFields": "OBJECTID", "f": "geojson"},
                  timeout=120).json()
boundary = gpd.GeoDataFrame.from_features(bj["features"], crs="EPSG:4326").to_crs(TARGET_CRS)
basin = boundary.geometry.unary_union
inside = culverts.geometry.within(basin)
clipped = culverts[~inside]
if len(clipped):
    clipped.drop(columns="geometry").to_csv(OUTPUTS / "clipped_out_of_basin.csv", index=False)
    log.warning(f"Clipped {len(clipped)} features outside the TRPA boundary "
                f"- see clipped_out_of_basin.csv:\n"
                + clipped["jurisdiction"].value_counts().to_string())
culverts = culverts[inside].reset_index(drop=True)
n_cond0 = len(condition)
condition = condition[condition["culvert_id"].isin(set(culverts["culvert_id"]))].reset_index(drop=True)
if n_cond0 - len(condition):
    log.info(f"Dropped {n_cond0 - len(condition)} condition records tied to clipped features")

log.info(f"Combined (clipped to TRPA boundary): {len(culverts)} culverts, {len(condition)} condition records")

2026-07-29 16:11:02 | WARNING | Clipped 1087 features outside the TRPA boundary - see clipped_out_of_basin.csv:
jurisdiction
Douglas County      854
Placer County       187
El Dorado County     43
Caltrans              3


2026-07-29 16:11:02 | INFO | Dropped 1044 condition records tied to clipped features


2026-07-29 16:11:02 | INFO | Combined (clipped to TRPA boundary): 6232 culverts, 2091 condition records


## Legacy TRPA compilation - gap fill (provisional)
`F:\...\PROTECT_analysis\Assets.gdb\Tahoe_Culvert` (read-only) is TRPA's previous basin-wide culvert compilation: 2,287 points merged from historical sources, including **USFS forest-road culverts** absent from every jurisdiction delivery. Each legacy point is matched to the nearest new asset within `run.legacy_match_m` metres; unmatched points are appended as jurisdiction **`TRPA Legacy (provisional)`** so the coverage gap is filled but the provenance is unmistakable. Legacy attributes are sparse (no universal ID; condition text on ~350 records) - IDs coalesce `CN` -> `NTCD_Asset` -> `PCID` -> row number, and Good/Fair/Poor condition text becomes provisional condition records.

In [10]:
LEGACY = Path(cfg["paths"]["legacy_culverts"])
MATCH_M = cfg["run"]["legacy_match_m"]

leg = gpd.read_file(LEGACY.parent, layer=LEGACY.name).to_crs(TARGET_CRS)
n0 = len(leg)
leg = leg[leg.geometry.notna() & leg.geometry.within(basin)].copy()
log.info(f"[Legacy] read {n0} points, {len(leg)} inside the TRPA boundary")

# Match to the nearest new asset within MATCH_M metres
near = gpd.sjoin_nearest(leg[["geometry"]], culverts[["culvert_id", "geometry"]],
                         how="left", max_distance=MATCH_M, distance_col="dist_m")
near = near[~near.index.duplicated(keep="first")]           # one match per legacy point
leg["matched_id"] = near["culvert_id"]
leg["dist_m"] = near["dist_m"]
missing = leg[leg["matched_id"].isna()].copy()
log.info(f"[Legacy] {len(leg) - len(missing)} matched an asset within {MATCH_M} m; "
         f"{len(missing)} are missing from the new layer")


def coalesce(df, cols):
    out = pd.Series(None, index=df.index, dtype=object)
    for c in cols:
        s = df[c].replace(" ", None).replace("", None) if df[c].dtype == object else df[c]
        out = out.fillna(s)
    return out


mid = coalesce(missing, ["CN", "NTCD_Asset"]).fillna(
    num(missing["PCID"]).astype("Int64").astype(str).replace("<NA>", None))
mid = mid.fillna(pd.Series("ROW" + missing.index.astype(str), index=missing.index))
missing["source_id"] = dedupe_ids(mid)

own = coalesce(missing, ["Ownership", "LANDOWNER_"]).fillna("unknown")
sizetxt = coalesce(missing, ["FEATURE_SI", "Pipe_Dimen"]).fillna("")

legacy_culverts = std_culverts(
    pd.DataFrame({
        "jurisdiction": "TRPA Legacy (provisional)",
        "source_id": missing["source_id"],
        "culvert_id": "TRPA Legacy (provisional)|" + missing["source_id"],
        "feature_type": "culvert",
        "road_name": coalesce(missing, ["Road_Name", "ROUTE_ID"]),
        "material": coalesce(missing, ["STR_MTR", "Pipe_Mater", "Materials", "Material"]),
        "xsection_shape": missing["SHAPE_TYPE"],
        "span_in": (num(missing["WIDTH_FT"]) * 12).fillna(num(missing["CUL_SIZE"])),
        "rise_in": num(missing["HEIGHT_FT"]) * 12,
        "comments": ("PROVISIONAL legacy record | Owner: " + own.astype(str)
                     + " | Size: " + sizetxt.astype(str)
                     + " | Stream: " + missing["STREAMNAME"].fillna("").astype(str)),
        "geom_source": "point",
        "data_source": "Assets.gdb/Tahoe_Culvert (legacy)",
        "load_date": LOAD_DATE,
    }),
    geometry=missing.geometry.apply(force_2d),
    src_crs=TARGET_CRS,
)

cond_txt = coalesce(missing, ["CONDITION", "PIPE_CONDI"]).replace({"N/A": None, "NC": None})
has_cond = cond_txt.notna()
legacy_condition = std_condition(pd.DataFrame({
    "culvert_id": ("TRPA Legacy (provisional)|" + missing.loc[has_cond, "source_id"]),
    "jurisdiction": "TRPA Legacy (provisional)",
    "inspection_date": pd.to_datetime(missing.loc[has_cond, "DATE_"], errors="coerce",
                                      utc=True).dt.tz_localize(None),
    "condition_rating": cond_txt[has_cond].str.split(" - ").str[0].str.title(),
    "condition_scheme": "TRPA legacy Good/Fair/Poor text (provisional, undated where blank)",
    "notes": cond_txt[has_cond],
    "data_source": "Assets.gdb/Tahoe_Culvert (legacy)",
    "load_date": LOAD_DATE,
}))

culverts = gpd.GeoDataFrame(pd.concat([culverts, legacy_culverts], ignore_index=True),
                            geometry="geometry", crs=TARGET_CRS)
condition = pd.concat([condition, legacy_condition], ignore_index=True)
log.info(f"[Legacy] appended {len(legacy_culverts)} provisional culverts, "
         f"{len(legacy_condition)} provisional condition records")

# Comparison report
leg_report = leg[["matched_id", "dist_m"]].copy()
leg_report["legacy_id"] = coalesce(leg, ["CN", "NTCD_Asset"]).fillna(
    num(leg["PCID"]).astype("Int64").astype(str).replace("<NA>", None))
leg_report["status"] = np.where(leg_report["matched_id"].notna(), "matched", "added_provisional")
leg_report.to_csv(OUTPUTS / "legacy_comparison.csv", index=False)
log.info("Wrote legacy_comparison.csv (per-legacy-point match status)")

2026-07-29 16:11:05 | INFO | [Legacy] read 2287 points, 2223 inside the TRPA boundary


2026-07-29 16:11:05 | INFO | [Legacy] 597 matched an asset within 25 m; 1626 are missing from the new layer


2026-07-29 16:11:06 | INFO | [Legacy] appended 1626 provisional culverts, 162 provisional condition records


2026-07-29 16:11:06 | INFO | Wrote legacy_comparison.csv (per-legacy-point match status)


## Persist
GeoPackage + parquet/CSV intermediates, written regardless of whether the GDB step can run.

In [11]:
log.info(f"Final: {len(culverts)} culverts, {len(condition)} condition records")
log.info("By jurisdiction:\n" + culverts["jurisdiction"].value_counts().to_string())
log.info("Feature type:\n" + culverts["feature_type"].value_counts().to_string())

gpkg = PROCESSED / "culverts.gpkg"
culverts.to_file(gpkg, layer="culverts", driver="GPKG")
cond_out = condition.copy()
cond_out.to_parquet(PROCESSED / "culvert_condition.parquet", index=False)
cond_out.to_csv(OUTPUTS / "culvert_condition.csv", index=False)
culverts.drop(columns="geometry").to_csv(OUTPUTS / "culverts_attributes.csv", index=False)
log.info(f"Wrote {gpkg}, culvert_condition.parquet/.csv, culverts_attributes.csv")

2026-07-29 16:11:06 | INFO | Final: 7858 culverts, 2253 condition records


2026-07-29 16:11:06 | INFO | By jurisdiction:
jurisdiction
El Dorado County             2615
Washoe County                2413
TRPA Legacy (provisional)    1626
Caltrans                      695
Placer County                 473
City of South Lake Tahoe       32
Douglas County                  4


2026-07-29 16:11:06 | INFO | Feature type:
feature_type
culvert            4268
stormwater pipe    3590


2026-07-29 16:11:09 | INFO | Wrote C:\Users\mbindl\Documents\GitHub\PROTECT\data\processed\culverts.gpkg, culvert_condition.parquet/.csv, culverts_attributes.csv


## Build the File GDB
Writes `Culverts` + `CulvertCondition` into `outputs/culverts.gdb` with a 1:many relationship class. Gated on `run.build_gdb` and an available arcpy license (run this from a machine/session where ArcGIS Pro can check out a license; the GeoPackage above is the fallback).

In [12]:
def _fields_spec(fields):
    """CULVERT/CONDITION field tuples -> AddFields spec [[name, type, alias, length], ...]."""
    return [[name, ftype, name, length if length else ""] for name, ftype, length in fields]


def _dt(v):
    return None if v is None or pd.isna(v) else pd.Timestamp(v).to_pydatetime()


# TEXT field lengths, for defensive truncation on insert (sources have verbose values)
TEXT_LEN = {name: length for name, ftype, length in CULVERT_FIELDS + CONDITION_FIELDS
            if ftype == "TEXT"}


def _clean(v, col=None):
    if v is None or (isinstance(v, float) and pd.isna(v)) or v is pd.NA:
        return None
    if isinstance(v, str) and col in TEXT_LEN and len(v) > TEXT_LEN[col]:
        return v[:TEXT_LEN[col]]
    return v


if not cfg["run"]["build_gdb"]:
    log.info("run.build_gdb is false - skipping GDB build")
elif not HAS_ARCPY:
    log.warning("arcpy unavailable - GDB not built. Use data/processed/culverts.gpkg, "
                "or re-run this cell where an ArcGIS license is available.")
else:
    arcpy.env.overwriteOutput = True
    gdb = str(OUTPUTS / cfg["output"]["gdb_name"])

    # Start from a clean GDB: a partial build from a prior attempt otherwise leaves
    # stale datasets and catalog-cache confusion (ERROR 000732 on just-created data).
    if arcpy.Exists(gdb):
        arcpy.management.Delete(gdb)
    arcpy.management.CreateFileGDB(str(OUTPUTS), cfg["output"]["gdb_name"])
    arcpy.management.ClearWorkspaceCache()

    sr = arcpy.SpatialReference(cfg["output"]["target_epsg"])

    # Use the tool Result paths rather than hand-built strings - avoids the
    # stale-catalog failure where AddField cannot see a just-created dataset.
    fc = arcpy.management.CreateFeatureclass(
        gdb, cfg["output"]["feature_class"], "POINT", spatial_reference=sr).getOutput(0)
    arcpy.management.AddFields(fc, _fields_spec(CULVERT_FIELDS))
    date_cols_c = {"load_date"}
    with arcpy.da.InsertCursor(fc, ["SHAPE@XY"] + CULVERT_COLS) as cur:
        for _, r in culverts.iterrows():
            xy = (r.geometry.x, r.geometry.y) if r.geometry is not None else None
            cur.insertRow([xy] + [_dt(r[c]) if c in date_cols_c else _clean(r[c], c)
                                  for c in CULVERT_COLS])
    log.info(f"Wrote {int(arcpy.management.GetCount(fc)[0])} rows -> {cfg['output']['feature_class']}")

    tbl = arcpy.management.CreateTable(gdb, cfg["output"]["condition_table"]).getOutput(0)
    arcpy.management.AddFields(tbl, _fields_spec(CONDITION_FIELDS))
    date_cols_t = {"inspection_date", "load_date"}
    with arcpy.da.InsertCursor(tbl, CONDITION_COLS) as cur:
        for _, r in condition.iterrows():
            cur.insertRow([_dt(r[c]) if c in date_cols_t else _clean(r[c], c)
                           for c in CONDITION_COLS])
    log.info(f"Wrote {int(arcpy.management.GetCount(tbl)[0])} rows -> {cfg['output']['condition_table']}")

    arcpy.management.CreateRelationshipClass(
        origin_table=fc, destination_table=tbl,
        out_relationship_class=f"{gdb}\\{cfg['output']['relationship_class']}",
        relationship_type="SIMPLE", forward_label="Condition", backward_label="Culvert",
        message_direction="NONE", cardinality="ONE_TO_MANY", attributed="NONE",
        origin_primary_key="culvert_id", origin_foreign_key="culvert_id")
    log.info(f"Relationship class created. GDB ready: {gdb}")

2026-07-29 16:12:03 | INFO | Wrote 7858 rows -> Culverts


2026-07-29 16:12:21 | INFO | Wrote 2253 rows -> CulvertCondition


2026-07-29 16:12:27 | INFO | Relationship class created. GDB ready: C:\Users\mbindl\Documents\GitHub\PROTECT\outputs\culverts.gdb


## QA
Key integrity, orphan condition records, missing-data rates, and spatial extent. Writes `outputs/qa_culverts.csv`. Report-only - review before publishing.

In [13]:
qa = {}
qa["culvert_count"] = len(culverts)
qa["condition_count"] = len(condition)
qa["null_culvert_id"] = int(culverts["culvert_id"].isna().sum())
qa["dup_culvert_id"] = int(culverts["culvert_id"].duplicated().sum())

known = set(culverts["culvert_id"])
qa["orphan_condition_rows"] = int((~condition["culvert_id"].isin(known)).sum())
qa["culverts_with_condition"] = int(condition["culvert_id"].nunique())
qa["culverts_without_condition"] = int(len(culverts) - culverts["culvert_id"].isin(set(condition["culvert_id"])).sum())

qa["null_geometry"] = int(culverts.geometry.isna().sum())
qa["null_material_pct"] = round(100 * culverts["material"].isna().mean(), 1)
qa["null_span_pct"] = round(100 * culverts["span_in"].isna().mean(), 1)
qa["null_inspection_date"] = int(condition["inspection_date"].isna().sum())

# Tahoe Basin bbox sanity check (UTM 10N, generous): lon -120.25..-119.85, lat 38.85..39.30
pts = culverts[culverts.geometry.notna()]
inside = pts.geometry.apply(lambda g: 730000 <= g.x <= 790000 and 4290000 <= g.y <= 4370000)
qa["outside_basin_bbox"] = int((~inside).sum())
if qa["outside_basin_bbox"]:
    out_rows = pts[~inside][["culvert_id", "jurisdiction", "road_name"]]
    out_rows.to_csv(OUTPUTS / "qa_outside_bbox.csv", index=False)
    log.warning(f"{qa['outside_basin_bbox']} culverts fall outside the basin bbox "
                f"- see qa_outside_bbox.csv (jurisdiction datasets may extend beyond the basin)")

for k, v in qa.items():
    log.info(f"QA {k}: {v}")
pd.json_normalize(qa).to_csv(OUTPUTS / "qa_culverts.csv", index=False)

# Per-jurisdiction summary for review
summary = (culverts.assign(has_geom=culverts.geometry.notna())
           .groupby("jurisdiction")
           .agg(culverts=("culvert_id", "count"),
                with_geom=("has_geom", "sum"),
                pct_material=("material", lambda s: round(100 * s.notna().mean(), 1)),
                pct_span=("span_in", lambda s: round(100 * s.notna().mean(), 1))))
summary["condition_records"] = condition.groupby("jurisdiction")["culvert_id"].count()
summary["condition_records"] = summary["condition_records"].fillna(0).astype(int)
summary.to_csv(OUTPUTS / "qa_by_jurisdiction.csv")
print(summary.to_string())

2026-07-29 16:12:27 | INFO | QA culvert_count: 7858


2026-07-29 16:12:27 | INFO | QA condition_count: 2253


2026-07-29 16:12:27 | INFO | QA null_culvert_id: 0


2026-07-29 16:12:27 | INFO | QA dup_culvert_id: 0


2026-07-29 16:12:27 | INFO | QA orphan_condition_rows: 0


...

2026-07-29 16:12:27 | INFO | QA null_material_pct: 22.4


2026-07-29 16:12:27 | INFO | QA null_span_pct: 21.9


2026-07-29 16:12:27 | INFO | QA null_inspection_date: 507


2026-07-29 16:12:27 | INFO | QA outside_basin_bbox: 0


                           culverts  with_geom  pct_material  pct_span  condition_records
jurisdiction                                                                             
Caltrans                        695        695          99.9      99.6                808
City of South Lake Tahoe         32         32         100.0     100.0                  0
Douglas County                    4          4         100.0     100.0                  4
El Dorado County               2615       2615          99.7      99.8                  0
Placer County                   473        473           0.8      84.6                473
TRPA Legacy (provisional)      1626       1626          22.4      18.3                162
Washoe County                  2413       2413          99.0      87.3                806


## Scope note: culverts vs. broader stormwater assets
For the PROTECT vulnerability assessment the asset of interest is the **culvert** - the road asset whose failure closes a route. Basins, inlets, manholes, and most conveyance pipe are water-quality infrastructure and are excluded (Douglas non-culvert features filtered out; CSLT pipes/channels/inlets not fetched). Where a source does not distinguish culverts from buried conveyance (El Dorado, non-crossing Washoe pipes), rows are kept but tagged `feature_type = 'stormwater pipe'` so the risk tool can filter to `feature_type = 'culvert'`.

## Open questions / next steps
1. **El Dorado domains**: get the coded-value lookups (`MATL_CODE`, `TYPE_SHAPE_CODE`, inlet/outlet treatment codes) - the delivered README is empty. Also confirm which pipes are culverts vs. buried conveyance.
2. **Condition semantics**: confirm direction of Placer's 1-5 rating and Washoe's structural/blockage codes before normalizing to a master condition scale.
3. **Line-to-point choice**: `run.line_to_point` is `midpoint`; flip to `start`/`end` in `config.yaml` if inlet ends are preferred.
4. **CSLT condition**: the service has no culvert inspection table - ask the City whether culvert inspections live elsewhere.
5. **NDOT**: pending the executed data sharing agreement.
6. **Provisional legacy records**: unmatched legacy points (largely USFS forest roads) are in as `TRPA Legacy (provisional)`. Field-verify or replace them as jurisdiction/USFS data improves; tune `run.legacy_match_m` if the 25 m match radius over- or under-merges.
7. **Publish**: load the GDB to SDE, publish culverts + condition as a REST service, and register it in `html/data-model-inventory.html`.

The full per-jurisdiction question list for data follow-ups lives in `docs/jurisdiction_data_questions.md`.